In [1]:
import json
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

/home/arch/.pyenv/versions/3.11.8/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14318.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
anchors = []
positives = []

In [4]:
with open('./templates/conditionSymptomMap.json', 'r') as f:
    condition_map = json.load(f)

for condition, symptoms in condition_map.items():
    for symptom in symptoms:
        anchors.append(symptom.lower())
        positives.append(condition)

In [5]:
with open('./dataset/clean_audiology_dataset.json', 'r') as f:
    cleaned_dataset = json.load(f)

for record in cleaned_dataset:
    text_input = record.get("cleaned_input", record.get("patient_input"))
    for condition in record["labels"]["possible_conditions"]:
        anchors.append(text_input)
        positives.append(condition)

In [6]:
len(anchors), len(positives)

(1613, 1613)

In [7]:
anchors[:5], positives[:5]

(['ear pain', 'ear itching', 'ear swelling', 'fluid discharge', 'ear redness'],
 ['Otitis Externa',
  'Otitis Externa',
  'Otitis Externa',
  'Otitis Externa',
  'Otitis Externa'])

In [8]:
trainDataset = Dataset.from_dict({
    "anchor": anchors,
    "positive": positives
})

In [9]:
loss = MultipleNegativesRankingLoss(model)

In [10]:
args = SentenceTransformerTrainingArguments(
    output_dir="./models/fine_tuned_audiology_sbert",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    warmup_steps=100,
    logging_steps=50,
)

In [11]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=trainDataset,
    loss=loss,
)

In [12]:
print("Starting modern fine-tuning...")
trainer.train()

Starting modern fine-tuning...


Step,Training Loss
50,2.910421
100,2.295851
150,2.057863
200,1.998360
250,1.879185
300,1.860924
350,1.807848
400,1.756080
450,1.698661
500,1.719860


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.10it/s]


TrainOutput(global_step=505, training_loss=1.9969054571472773, metrics={'train_runtime': 25.1427, 'train_samples_per_second': 320.769, 'train_steps_per_second': 20.085, 'total_flos': 0.0, 'train_loss': 1.9969054571472773, 'epoch': 5.0})